# Gas Price Data Validation — PJM Like-Day Pipeline

**Source:** `ice.next_day_gas` WHERE hub IN ('M3', 'HH', 'Transco Z6 NY') AND data_type='VWAP Close'  
**Format:** Pivoted wide — `date, gas_m3_price, gas_hh_price, gas_transco_z6_price`  
**Purpose:** Validate raw gas price data before it enters the like-day feature pipeline.

### Checks
1. **Completeness** — Date range, trading-day coverage, gap analysis
2. **Nulls & Outliers** — Per-hub null counts, z-score outliers, physical bounds
3. **Distributions** — Histograms for M3, HH, and M3-HH spread; monthly boxplots
4. **Price Time Series** — All hubs over time with regime highlights
5. **Basis Spread Analysis** — M3-HH spread dynamics, monthly patterns, hub correlation
6. **Temporal Continuity** — Day-over-day changes, extreme moves, rolling mean vs spot
7. **Feature Sanity Check** — Verify derived features from `gas_features.build()`
8. **Recent Spot Check** — Last 14 days table, last 30 days chart
9. **Validation Summary** — Pass/fail checklist

## 1. Setup & Data Pull

In [ ]:
import sys
from pathlib import Path
_BACKEND = str(Path().resolve().parent.parent.parent.parent)
if _BACKEND not in sys.path:
    sys.path.insert(0, _BACKEND)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.like_day_forecast.settings
from src.data import gas_prices
from src.like_day_forecast import configs

In [ ]:
# Pull gas price data
df_gas = gas_prices.pull()
df_gas["date"] = pd.to_datetime(df_gas["date"])

print(f"Shape: {df_gas.shape}")
print(f"Date range: {df_gas['date'].min().date()} to {df_gas['date'].max().date()}")
print(f"Unique dates: {df_gas['date'].nunique():,}")
print(f"\nColumns: {list(df_gas.columns)}")
print(f"\nDtypes:\n{df_gas.dtypes}")
df_gas.head()

### Previous 3 Days — M3, HH, Transco Z6, and M3-HH Spread

Quick visual inspection of the most recent 3 trading days across all gas price components.

In [ ]:
# Previous 3 trading days — all gas components
df_last3 = df_gas.sort_values("date").tail(3).copy()
df_last3["m3_hh_spread"] = df_last3["gas_m3_price"] - df_last3["gas_hh_price"]

print("=== M3 (TETCO) — Previous 3 Trading Days ===")
display(df_last3[["date", "gas_m3_price"]])

print("\n=== Henry Hub (HH) — Previous 3 Trading Days ===")
display(df_last3[["date", "gas_hh_price"]])

print("\n=== Transco Z6 NY — Previous 3 Trading Days ===")
display(df_last3[["date", "gas_transco_z6_price"]])

print("\n=== M3-HH Basis Spread — Previous 3 Trading Days ===")
display(df_last3[["date", "gas_m3_price", "gas_hh_price", "m3_hh_spread"]].round(4))

## 2. Completeness Checks

Gas does not trade on weekends or federal holidays, so gaps on those days are expected. We check for unexpected mid-week gaps, duplicates, and overall trading-day coverage.

In [ ]:
# 2a. Calendar days vs trading days
all_dates = pd.date_range(df_gas["date"].min(), df_gas["date"].max(), freq="D")
present_dates = set(df_gas["date"].dt.normalize().unique())
missing_dates = sorted(set(all_dates) - present_dates)

# Separate weekend vs weekday gaps
missing_weekend = [d for d in missing_dates if d.weekday() >= 5]
missing_weekday = [d for d in missing_dates if d.weekday() < 5]

print(f"Calendar days in range: {len(all_dates):,}")
print(f"Trading days present:   {len(present_dates):,}")
print(f"Missing calendar days:  {len(missing_dates):,}")
print(f"  - Weekend/expected:   {len(missing_weekend):,}")
print(f"  - Weekday (holidays): {len(missing_weekday):,}")

# 2b. Identify longest gap
df_sorted = df_gas.sort_values("date").copy()
df_sorted["gap_days"] = df_sorted["date"].diff().dt.days
longest_gap_idx = df_sorted["gap_days"].idxmax()
longest_gap = df_sorted.loc[longest_gap_idx]
print(f"\nLongest gap: {int(longest_gap['gap_days'])} calendar days ending {longest_gap['date'].date()}")

# Show top 10 longest gaps
print("\nTop 10 longest gaps:")
top_gaps = df_sorted.nlargest(10, "gap_days")[["date", "gap_days"]].copy()
top_gaps["date"] = top_gaps["date"].dt.date
display(top_gaps)

# 2c. Check for duplicate dates
dupes = df_gas.duplicated(subset=["date"], keep=False)
print(f"\nDuplicate dates: {dupes.sum()}")
if dupes.sum() > 0:
    display(df_gas[dupes].sort_values("date"))

# 2d. Show weekday missing dates (likely holidays)
if len(missing_weekday) > 0:
    print(f"\nWeekday gaps (likely holidays) — showing up to 30:")
    for d in missing_weekday[:30]:
        print(f"  {d.date()} ({d.strftime('%A')})")
    if len(missing_weekday) > 30:
        print(f"  ... and {len(missing_weekday) - 30} more")

## 3. Null & Outlier Analysis

In [ ]:
price_cols = ["gas_m3_price", "gas_hh_price", "gas_transco_z6_price"]

# Null summary per hub
null_summary = pd.DataFrame({
    "null_count": df_gas[price_cols].isnull().sum(),
    "null_pct": (df_gas[price_cols].isnull().sum() / len(df_gas) * 100).round(3),
    "non_null": df_gas[price_cols].notnull().sum(),
})
print("Null Summary per Hub:")
display(null_summary)

# Descriptive statistics
print("\nDescriptive Statistics:")
display(df_gas[price_cols].describe().round(4))

# Z-score outlier detection (|z| > 4)
print("\nOutlier Check (|z| > 4):")
for col in price_cols:
    series = df_gas[col].dropna()
    z = (series - series.mean()) / series.std()
    n_outliers = (z.abs() > 4).sum()
    print(f"  {col}: {n_outliers} outliers  (range [{series.min():.2f}, {series.max():.2f}])")
    if n_outliers > 0 and n_outliers <= 10:
        outlier_dates = df_gas.loc[z[z.abs() > 4].index, ["date", col]]
        outlier_dates["date"] = outlier_dates["date"].dt.date
        print(f"    Outlier dates: {list(outlier_dates.itertuples(index=False, name=None))}")

# Physical bounds check (gas price 0-50 normal, flag >$100)
print("\n--- Physical Bounds ---")
for col in price_cols:
    series = df_gas[col].dropna()
    n_negative = (series < 0).sum()
    n_over_50 = (series > 50).sum()
    n_over_100 = (series > 100).sum()
    print(f"  {col}:")
    print(f"    Negative: {n_negative}")
    print(f"    > $50/MMBtu: {n_over_50}")
    print(f"    > $100/MMBtu: {n_over_100}")

## 4. Distributions

In [ ]:
# Histograms for M3, HH, and M3-HH spread
df_gas["gas_m3_hh_spread"] = df_gas["gas_m3_price"] - df_gas["gas_hh_price"]

fig = make_subplots(rows=1, cols=3, subplot_titles=["M3 Price", "HH Price", "M3-HH Spread"])

fig.add_trace(
    go.Histogram(x=df_gas["gas_m3_price"].dropna(), nbinsx=80, name="M3", showlegend=False,
                 marker_color="steelblue"),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=df_gas["gas_hh_price"].dropna(), nbinsx=80, name="HH", showlegend=False,
                 marker_color="darkorange"),
    row=1, col=2,
)
fig.add_trace(
    go.Histogram(x=df_gas["gas_m3_hh_spread"].dropna(), nbinsx=80, name="Spread", showlegend=False,
                 marker_color="mediumseagreen"),
    row=1, col=3,
)

fig.update_xaxes(title_text="$/MMBtu", row=1, col=1)
fig.update_xaxes(title_text="$/MMBtu", row=1, col=2)
fig.update_xaxes(title_text="$/MMBtu", row=1, col=3)
fig.update_layout(height=400, title_text="Gas Price Distributions")
fig.show()

In [ ]:
# Monthly boxplot of M3 price — seasonal pattern check
df_gas["month"] = df_gas["date"].dt.month
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
df_gas["month_name"] = df_gas["month"].map(month_names)

fig = px.box(df_gas, x="month_name", y="gas_m3_price",
             category_orders={"month_name": list(month_names.values())},
             title="Monthly M3 Gas Price Distribution (all years)",
             labels={"gas_m3_price": "M3 Price ($/MMBtu)", "month_name": "Month"})
fig.update_layout(height=450)
fig.show()

## 5. Price Time Series

M3, HH, and Transco Z6 NY prices over time. Key regimes:
- **Pre-COVID (2014-2019):** Low, stable gas prices with seasonal winter spikes
- **COVID crash (Mar-May 2020):** Demand destruction, prices near all-time lows
- **2021-2022 gas spike:** Supply-constrained, Russia/Ukraine, prices >$8+

In [ ]:
# All hubs over time
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_gas["date"], y=df_gas["gas_m3_price"],
    mode="lines", name="M3 (TETCO)", line=dict(width=0.8, color="steelblue"),
))
fig.add_trace(go.Scatter(
    x=df_gas["date"], y=df_gas["gas_hh_price"],
    mode="lines", name="Henry Hub", line=dict(width=0.8, color="darkorange"),
))
fig.add_trace(go.Scatter(
    x=df_gas["date"], y=df_gas["gas_transco_z6_price"],
    mode="lines", name="Transco Z6 NY", line=dict(width=0.8, color="mediumseagreen"),
))

# Regime shading
fig.add_vrect(x0="2020-03-01", x1="2020-06-01", fillcolor="red", opacity=0.08,
              annotation_text="COVID", annotation_position="top left")
fig.add_vrect(x0="2021-09-01", x1="2022-09-01", fillcolor="orange", opacity=0.08,
              annotation_text="Gas Spike", annotation_position="top left")

fig.update_layout(
    title="Next-Day Gas Prices — M3, HH, Transco Z6 NY",
    xaxis_title="Date",
    yaxis_title="Price ($/MMBtu)",
    height=500,
    hovermode="x unified",
)
fig.show()

## 6. Basis Spread Analysis

M3 is the PJM-relevant gas hub. The M3-HH spread captures the regional basis premium driven by pipeline constraints, especially during winter cold snaps.

In [ ]:
# M3-HH spread time series
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_gas["date"], y=df_gas["gas_m3_hh_spread"],
    mode="lines", name="M3-HH Spread",
    line=dict(width=0.8, color="mediumseagreen"),
))
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(
    title="M3 - HH Basis Spread Over Time",
    xaxis_title="Date",
    yaxis_title="Spread ($/MMBtu)",
    height=400,
    hovermode="x unified",
)
fig.show()

spread = df_gas["gas_m3_hh_spread"].dropna()
print(f"M3-HH Spread: mean={spread.mean():.3f}, median={spread.median():.3f}, "
      f"std={spread.std():.3f}")
print(f"  Min: {spread.min():.2f} on {df_gas.loc[spread.idxmin(), 'date'].date()}")
print(f"  Max: {spread.max():.2f} on {df_gas.loc[spread.idxmax(), 'date'].date()}")
print(f"  Negative spread days: {(spread < 0).sum()} ({(spread < 0).mean()*100:.1f}%)")

In [ ]:
# Monthly spread boxplot
fig = px.box(df_gas, x="month_name", y="gas_m3_hh_spread",
             category_orders={"month_name": list(month_names.values())},
             title="Monthly M3-HH Basis Spread Distribution",
             labels={"gas_m3_hh_spread": "M3-HH Spread ($/MMBtu)", "month_name": "Month"})
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(height=450)
fig.show()

In [ ]:
# M3 vs HH scatter — color by year
df_gas["year"] = df_gas["date"].dt.year

fig = px.scatter(
    df_gas.dropna(subset=["gas_m3_price", "gas_hh_price"]),
    x="gas_hh_price", y="gas_m3_price", color="year",
    title="M3 vs HH Price — Colored by Year",
    labels={"gas_hh_price": "HH Price ($/MMBtu)", "gas_m3_price": "M3 Price ($/MMBtu)"},
    opacity=0.5,
    color_continuous_scale="Viridis",
)
# Add 1:1 reference line
max_val = max(df_gas["gas_m3_price"].max(), df_gas["gas_hh_price"].max())
fig.add_trace(go.Scatter(
    x=[0, max_val], y=[0, max_val], mode="lines",
    line=dict(dash="dot", color="gray"), name="1:1 Line", showlegend=True,
))
fig.update_layout(height=500)
fig.show()

# Correlation by year
print("M3 vs HH correlation by year:")
for yr, grp in df_gas.dropna(subset=["gas_m3_price", "gas_hh_price"]).groupby("year"):
    r = grp[["gas_m3_price", "gas_hh_price"]].corr().iloc[0, 1]
    print(f"  {yr}: r = {r:.3f}  (n={len(grp)})")

## 7. Temporal Continuity

Day-over-day M3 price changes. Extreme moves (>$5/MMBtu) flagged. Rolling 30-day mean vs spot to detect drift or data issues.

In [ ]:
# Day-over-day M3 change
df_gas_sorted = df_gas.sort_values("date").copy()
df_gas_sorted["m3_daily_change"] = df_gas_sorted["gas_m3_price"].diff()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["M3 Spot Price", "Day-over-Day Change"])

fig.add_trace(go.Scatter(
    x=df_gas_sorted["date"], y=df_gas_sorted["gas_m3_price"],
    mode="lines", line=dict(width=0.8, color="steelblue"), name="M3 Spot"),
    row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_gas_sorted["date"], y=df_gas_sorted["m3_daily_change"],
    mode="lines", line=dict(width=0.8, color="steelblue"), name="Daily Change"),
    row=2, col=1)

# Flag extreme moves (>$5/MMBtu)
extreme_mask = df_gas_sorted["m3_daily_change"].abs() > 5
if extreme_mask.sum() > 0:
    fig.add_trace(go.Scatter(
        x=df_gas_sorted.loc[extreme_mask, "date"],
        y=df_gas_sorted.loc[extreme_mask, "m3_daily_change"],
        mode="markers", marker=dict(color="red", size=6),
        name=f"Extreme (>$5): {extreme_mask.sum()}"),
        row=2, col=1)

fig.update_layout(height=600, showlegend=True)
fig.update_yaxes(title_text="$/MMBtu", row=1, col=1)
fig.update_yaxes(title_text="Change ($/MMBtu)", row=2, col=1)
fig.show()

change = df_gas_sorted["m3_daily_change"].dropna()
print(f"M3 day-over-day change: mean={change.mean():.4f}, std={change.std():.4f}")
print(f"  Min: ${change.min():.2f} on {df_gas_sorted.loc[change.idxmin(), 'date'].date()}")
print(f"  Max: ${change.max():.2f} on {df_gas_sorted.loc[change.idxmax(), 'date'].date()}")
print(f"  |change| > $2: {(change.abs() > 2).sum()} days")
print(f"  |change| > $5: {(change.abs() > 5).sum()} days")
print(f"  |change| > $10: {(change.abs() > 10).sum()} days")

# Show extreme moves
if extreme_mask.sum() > 0:
    print(f"\nExtreme M3 moves (>$5/MMBtu):")
    display(df_gas_sorted.loc[extreme_mask, ["date", "gas_m3_price", "m3_daily_change"]]
            .sort_values("m3_daily_change", key=abs, ascending=False).head(20))

In [ ]:
# Rolling 30-day mean vs spot
df_gas_sorted["m3_30d_mean"] = df_gas_sorted["gas_m3_price"].rolling(30, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_gas_sorted["date"], y=df_gas_sorted["gas_m3_price"],
    mode="lines", name="M3 Spot", line=dict(width=0.6, color="steelblue"),
))
fig.add_trace(go.Scatter(
    x=df_gas_sorted["date"], y=df_gas_sorted["m3_30d_mean"],
    mode="lines", name="30-Day Rolling Mean", line=dict(width=1.5, color="firebrick"),
))
fig.update_layout(
    title="M3 Spot vs 30-Day Rolling Mean",
    xaxis_title="Date",
    yaxis_title="Price ($/MMBtu)",
    height=450,
    hovermode="x unified",
)
fig.show()

## 8. Feature Sanity Check

Run `gas_features.build()` and verify derived features match expectations.

In [ ]:
from src.like_day_forecast.features import gas_features

df_feat = gas_features.build(df_gas)
df_feat["date"] = pd.to_datetime(df_feat["date"])

print(f"Feature DataFrame shape: {df_feat.shape}")
print(f"Columns: {list(df_feat.columns)}")
print(f"\nDescriptive Statistics:")
display(df_feat.describe().round(4))

In [ ]:
# Verify spread formula: gas_m3_hh_spread = gas_m3_price - gas_hh_price
spread_expected = df_feat["gas_m3_price"] - df_feat["gas_hh_price"]
spread_match = np.isclose(df_feat["gas_m3_hh_spread"], spread_expected, atol=1e-6, equal_nan=True)
n_mismatch = (~spread_match).sum()

print("Spread Formula Verification:")
print(f"  gas_m3_hh_spread == gas_m3_price - gas_hh_price: {spread_match.all()}")
print(f"  Mismatches: {n_mismatch}")

if n_mismatch > 0:
    print("\n  Mismatched rows:")
    display(df_feat[~spread_match][["date", "gas_m3_price", "gas_hh_price", "gas_m3_hh_spread"]].head(10))

# Null counts in features
print("\nNull counts in features:")
feat_cols = [c for c in df_feat.columns if c != "date"]
for col in feat_cols:
    n_null = df_feat[col].isnull().sum()
    pct = n_null / len(df_feat) * 100
    print(f"  {col}: {n_null} ({pct:.1f}%)")

In [ ]:
# Feature correlation heatmap
corr = df_feat[feat_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Gas Feature Correlation Matrix",
    aspect="auto",
)
fig.update_layout(height=500, width=600)
fig.show()

# Flag highly correlated pairs
print("Highly correlated feature pairs (|r| > 0.90):")
for i in range(len(feat_cols)):
    for j in range(i + 1, len(feat_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.90:
            print(f"  {feat_cols[i]} <-> {feat_cols[j]}: r = {r:.3f}")

## 9. Recent Spot Check

In [ ]:
# Last 14 trading days table
recent_14 = df_gas.sort_values("date").tail(14)[
    ["date", "gas_m3_price", "gas_hh_price", "gas_transco_z6_price", "gas_m3_hh_spread"]
].copy()
recent_14["date"] = recent_14["date"].dt.strftime("%Y-%m-%d (%a)")

print("Last 14 Trading Days — Gas Prices")
display(recent_14)

In [ ]:
# Last 30 trading days — M3 price line chart
recent_30 = df_gas.sort_values("date").tail(30).copy()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recent_30["date"], y=recent_30["gas_m3_price"],
    mode="lines+markers", name="M3",
    line=dict(width=1.5, color="steelblue"),
    marker=dict(size=4),
))
fig.update_layout(
    title="M3 Gas Price — Last 30 Trading Days",
    xaxis_title="Date",
    yaxis_title="Price ($/MMBtu)",
    height=400,
    hovermode="x unified",
)
fig.show()

## 10. Validation Summary

In [ ]:
# Compute validation inputs
n_total = len(df_gas)
date_min = df_gas["date"].min().date()
date_max = df_gas["date"].max().date()
n_dupes = df_gas.duplicated(subset=["date"], keep=False).sum()

# Null percentages per hub
null_pct_m3 = df_gas["gas_m3_price"].isnull().sum() / n_total * 100
null_pct_hh = df_gas["gas_hh_price"].isnull().sum() / n_total * 100
null_pct_tz6 = df_gas["gas_transco_z6_price"].isnull().sum() / n_total * 100
max_null_pct = max(null_pct_m3, null_pct_hh, null_pct_tz6)

# M3 bounds
m3_min = df_gas["gas_m3_price"].min()
m3_max = df_gas["gas_m3_price"].max()

# Spread formula
spread_formula_ok = spread_match.all()

# Extreme single-day moves
m3_change = df_gas.sort_values("date")["gas_m3_price"].diff()
n_extreme_20 = (m3_change.abs() > 20).sum()

checks = [
    ("Date range covers 2014 (EXTENDED_FEATURE_START)",
     date_min <= pd.Timestamp("2014-01-01").date(),
     f"Range: {date_min} to {date_max}"),

    ("No duplicate dates",
     n_dupes == 0,
     f"{n_dupes} duplicates found"),

    ("Nulls < 5% per hub (weekends/holidays expected in calendar, not in trading data)",
     max_null_pct < 5,
     f"M3: {null_pct_m3:.1f}%, HH: {null_pct_hh:.1f}%, Transco Z6: {null_pct_tz6:.1f}%"),

    ("M3 price in bounds (0-100 $/MMBtu)",
     m3_min >= 0 and m3_max <= 100,
     f"Range: [{m3_min:.2f}, {m3_max:.2f}]"),

    ("M3-HH spread formula verified (gas_m3_price - gas_hh_price)",
     spread_formula_ok,
     f"All {len(df_feat):,} rows match" if spread_formula_ok else f"{n_mismatch} mismatches"),

    ("No extreme single-day M3 moves (> $20/MMBtu)",
     n_extreme_20 == 0,
     f"{n_extreme_20} extreme moves found"),
]

print("=" * 80)
print("  GAS PRICE DATA VALIDATION SUMMARY")
print("=" * 80)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")
    print(f"         {detail}")
print("=" * 80)
print(f"  {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED — review details above'}")
print("=" * 80)